In [1]:
from dataclasses import dataclass
from pathlib import Path

import jax

# Enable double precision before creating JAX arrays
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import yaml

from fmmax import basis, fields, fmm, scattering, utils

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


In [2]:
@dataclass
class SquareParticleModel:
    """Physical description of one periodic square particle structure.

    All lengths are expressed in nanometers. Angles are expressed in degrees because
    that is what S4 expects. 

    Attributes
    ----------
    period_nm: 
        Period of the square lattice in both x and y
    particle_side_nm:
        Full side length of the square particle.
    planarization_thickness_nm:
        Thickness of the homogeneous planarization layer.
    patterned_layer_thickness_nm:
        Thickness of the layer containing the square particle. 
    wavelength_nm:
        Free-space wavelength.
    theta_deg:
        Polar incidence angle measured from the surface normal.
    phi_deg:
        Azimuthal incidence angle.
    incidence_epsilon:
        Permittivity of the semi-infinite incidence material
    planarization_epsilon:
        Permittivity of the homogeneous planarization layer
    background_epsilon:
        Permittivity of the material surrounding the square particle
    particle_epsilon:
        Permittivity of the square particle.
    transsmission_epsilon:
        Permittivity of the semi-infinite transmission material
    """

    period_nm: float
    particle_side_nm: float
    planarization_thickness_nm: float
    patterned_layer_thickness_nm: float
    wavelength_nm: float
    theta_deg: float
    phi_deg: float
    incidence_epsilon: complex
    planarization_epsilon: complex
    background_epsilon: complex
    particle_epsilon: complex
    transmission_epsilon: complex

    @classmethod
    def from_dict(cls, data:dict) -> "SquareParticleModel":
        """Create a physical model from YAML-derived dictionary data."""

        # Copy the dictionary so we do not modify the original data
        data = dict(data)

        epsilon_list = (
            "incidence_epsilon",
            "planarization_epsilon",
            "background_epsilon", 
            "particle_epsilon",
            "transmission_epsilon",
        )

        # Convert each {real:..., image:...} dictionary into 
        # an ordinary Python complex number
        for epsilon in epsilon_list:
            components = data[epsilon]

            data[epsilon] = complex(
                components["real"],
                components.get("imag",0.0)
            )

        return cls(**data)

In [3]:
TRUNCATION_MAP = {
    "Circular": basis.Truncation.CIRCULAR,
    "Parallelogramic": basis.Truncation.PARALLELOGRAMIC,
}

FORMULATION_MAP = {
    "FFT": fmm.Formulation.FFT,
}

In [4]:
@dataclass
class FMMaxConfig:
    """Numerical settings used directly by FMMax.

    Attributes
    ---------
    nx, ny:
        Number of real-space samples used to represent the unit cell.
    approximate_num_terms:
        Approximate number of Fourier orders requested from FMMax.
    truncation:
        Rule used to select Fourier orders.
    formulation:
        Fourier-modal formulation used for patterned layers.
    """

    nx: int
    ny: int
    approximate_num_terms: int
    truncation: str
    formulation: str

    def __post_init__(self) -> None:
        """Check that the numerical settings are valid."""

        if self.nx <= 0 or self.ny <=0:
            raise ValueError(
                "nx and ny must be positive"
            )
        if self.approximate_num_terms <=0:
            raise ValueError(
                "approximate_num_terms must be positive"
            )
        allowed_truncation = (
            "Circular",
            "Parallelogramic"
        )

        if self.truncation not in allowed_truncation:
            raise ValueError(
                "truncation must be "
                "'Circular' or 'Parallelogramic'"

            )
        allowed_formulations = {"FFT"}

        if self.formulation not in allowed_formulations:
            raise ValueError(
                "The first notebook supports formulation='FFT'"

            )

    @classmethod
    def from_dict(cls, data:dict) -> "FMMaxConfig":
        """Create FMMax settings from YAML-derived dictionary data"""

        return cls(**data)

In [5]:
config_path = Path("../src/configs/square_particle.yaml")

with config_path.open("r") as file:
    yaml_data = yaml.safe_load(file)

model = SquareParticleModel.from_dict(
    yaml_data["model"]
)

config = FMMaxConfig.from_dict(
    yaml_data["fmmax"]
)

In [6]:
def rasterise_square(
        model: SquareParticleModel,
        config: FMMaxConfig
):
    """Rasterise the square particle over one periodic unit cell.

    The analytical square is sampled on an `nx` by `ny` grid. FMMax
    uses the resulting density mask to contruct the patterned layer's
    permittivity distribution.

    A density value of 1 represents particle material, while 0
    represents background material.

    Parameters
    ----------
    model:
        Physical square particle source and structure
    config:
        Numerical settings used by FMMax
    
    Returns
    -------
    x_grid, y_grid:
        Coordinates of the sampled points, each with shape (nx,ny)
    density:
        Binary particle mask with shape (nx,ny)
    """
    # First rasterise space
    # Sample one square unit cell over [-period/2, period/2]
    # with nx and ny representing the number of samples along x and y

    # Calculate the steps size between start and stop interval
    # Will use arange instead of linspace in order to not duplicate the
    # same periodic boundary
    dx_nm = model.period_nm / config.nx
    dy_nm = model.period_nm / config.ny

    x_grid,y_grid = jnp.meshgrid(
        jnp.arange(-model.period_nm/2, model.period_nm/2, dx_nm),
        jnp.arange(-model.period_nm/2, model.period_nm/2, dy_nm),
        indexing="ij"
    )

    # Assign 1 to the sampled spaces that is within the square
    # Assign 0 everywhere else 
    # This will be used to create the patterned layer
    inside_square_x = jnp.abs(x_grid) <= model.particle_side_nm / 2
    inside_square_y = jnp.abs(y_grid) <= model.particle_side_nm / 2

    density = (inside_square_x & inside_square_y).astype(float)

    return x_grid, y_grid, density
    


In [7]:
# List the different permittivities

x_grid, y_grid, density = rasterise_square(
    model=model,
    config=config
)

permittivities = [
    jnp.asarray([[model.incidence_epsilon]]),
    jnp.asarray([[model.planarization_epsilon]]),
    utils.interpolate_permittivity(
        permittivity_solid=jnp.asarray(model.particle_epsilon),
        permittivity_void=jnp.asarray(model.background_epsilon),
        density=density
    ),
    jnp.asarray([[model.transmission_epsilon]])
]

In [8]:
# The thicknesses of each layer
# The order has to match the order in permittivities:
# incidence, planarization, patterned layer, transmission
# Zero thickness represents the semi-infinite layer on either side
# of the structure
thicknesses = [0,
               model.planarization_thickness_nm,
               model.patterned_layer_thickness_nm,
               0]

In [9]:
# Create the reciprocal space Fourier basis used by FMMax
# The real space lattice vectors determine the possible
# periodic diffraction orders. approximate_num_terms
# requests the approximate number of order to retain, and 
# truncation determines how these orders are selected.
primitive_lattice_vectors=basis.LatticeVectors(
        u=jnp.asarray([model.period_nm, 0.0]), 
        v = jnp.asarray([0.0, model.period_nm]))

expansion = basis.generate_expansion(
    primitive_lattice_vectors=primitive_lattice_vectors,
        approximate_num_terms = config.approximate_num_terms,
        truncation=TRUNCATION_MAP[config.truncation]
    )

In [10]:
# Calculate the in-plane wavevector, but need to convert
# the polar and azimuthal angles to radians from degrees
# as our YAML files is in degrees and FMMax accepts radians

polar_angle = jnp.deg2rad(model.theta_deg)
azimuthal_angle = jnp.deg2rad(model.phi_deg)

in_plane_wavevector = basis.plane_wave_in_plane_wavevector(
    wavelength = jnp.asarray(model.wavelength_nm),
    polar_angle = polar_angle,
    azimuthal_angle=azimuthal_angle,
    permittivity=model.incidence_epsilon)

# Solves Maxwell's equations seperately inside every layer of
# our structure.
# index[0] -> incidence medium, index[1] -> planarization layer, 
# index[2] -> patterned layer, index[3] -> transmission medium
layer_solve_results = [
    fmm.eigensolve_isotropic_media(
        wavelength=jnp.asarray(model.wavelength_nm),
        in_plane_wavevector=in_plane_wavevector,
        primitive_lattice_vectors=primitive_lattice_vectors,
        permittivity=permittivity,
        expansion=expansion,
        formulation=FORMULATION_MAP[config.formulation]
    )
    for permittivity in permittivities
]


In [11]:
# Combine the independently solved layers into one scattering matrix.
# This accounts for propagation through each thickness and coupling
# between modes at every material interface.

s_matrix = scattering.stack_s_matrix(
    layer_solve_results = layer_solve_results,
    layer_thicknesses=[
        jnp.asarray(thickness) 
        for thickness in thicknesses]
)

In [12]:
def reflectance_and_transmittance_fmmax(
        s_matrix,
        layer_solve_results,
        polarization: str,
        thicknesses
):
    # Number of retained diffarction orders
    # FMMax stores two modal channels per diffraction order
    # index 0 has the first polarization block and index 1 the second
    n = layer_solve_results[0].expansion.num_terms

    # Zeroth-order plane wave excitation for s and p polarisation
    # First axis, axis 0: modal channel
    # which is diffraction order and polarisation block of output
    # Second axis, axis 1: independent incident source polarisation
    forward_amplitude_0 = jnp.zeros((2*n,2), dtype=complex)
    forward_amplitude_0 = forward_amplitude_0.at[0,0].set(1) #s
    forward_amplitude_0 = forward_amplitude_0.at[n,1].set(1) #p

    # Reflected zeroth order amplitude
    # s21 maps incident amplitudes to reflected amplitudes
    reflected_amplitude_0 = s_matrix.s21 @ forward_amplitude_0

    # Incident and reflected power flux by modal channel
    incident_flux, reflected_flux = fields.amplitude_poynting_flux(
        forward_amplitude=forward_amplitude_0,
        backward_amplitude=reflected_amplitude_0,
        layer_solve_result=layer_solve_results[0]
    )

    # s11 maps incident amplitudes to transmitted amplitudes
    transmitted_amplitude_0 = s_matrix.s11 @ forward_amplitude_0

    # No wave is incident from the transmission side
    transmitted_flux,_=fields.amplitude_poynting_flux(
        forward_amplitude=transmitted_amplitude_0,
        backward_amplitude=jnp.zeros_like(transmitted_amplitude_0),
        layer_solve_result=layer_solve_results[-1]
    )

    # Sum over every diffraction order and output polarisation
    incident_power = jnp.sum(incident_flux, axis=0)
    reflected_power = jnp.sum(reflected_flux, axis=0)
    transmitted_power = jnp.sum(transmitted_flux, axis=0)

    # Total reflectance for s incidence would be R[0]
    # Total reflectance for p incidence would be R[1]
    R = -reflected_power / incident_power 
    T = transmitted_power / incident_power

    return{
        "R": R,
        "T": T
    }

In [13]:
result = reflectance_and_transmittance_fmmax(
    s_matrix=s_matrix,
    layer_solve_results=layer_solve_results,
    polarization="s",
    thicknesses=thicknesses,
)

R = result["R"]
T = result["T"]

print(f"Rs = {R[0]:.10f}")
print(f"Rp = {R[1]:.10f}")
print(f"Ts = {T[0]:.10f}")
print(f"Tp = {T[1]:.10f}")
print("FMMax terms:", expansion.num_terms)

Rs = 0.0208197939
Rp = 0.0208197939
Ts = 0.9791802061
Tp = 0.9791802061
FMMax terms: 21


In [14]:
from dataclasses import replace
from pathlib import Path

import csv

requested_orders = list(range(5,101,5))

print(requested_orders)

[5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]


In [15]:
fmmax_rs_values = []
fmmax_actual_orders = []

for requested_num_terms in requested_orders:

    # Copy the original FMMax configuration while changing only
    # the requested Fourier-basis size

    sweep_config = replace(config,
                           approximate_num_terms = requested_num_terms)

    # Generate the new FMMax Fourier expansion
    sweep_expansion = basis.generate_expansion(
        primitive_lattice_vectors=primitive_lattice_vectors,
        approximate_num_terms = sweep_config.approximate_num_terms,
        truncation=TRUNCATION_MAP[sweep_config.truncation]
        )

    # Resolve the electromagnetic modes in every layer using the new
    # Fourier expansion
    sweep_layer_solve_result = [
        fmm.eigensolve_isotropic_media(
            wavelength=jnp.asarray(model.wavelength_nm),
            in_plane_wavevector=in_plane_wavevector,
            primitive_lattice_vectors=primitive_lattice_vectors,
            permittivity=permittivity,
            expansion=sweep_expansion,
            formulation=FORMULATION_MAP[sweep_config.formulation]
        )
        for permittivity in permittivities
    ]

    # Combine the layers into the complex scattering matrix
    sweep_s_matrix = scattering.stack_s_matrix(
        layer_solve_results=(
            sweep_layer_solve_result
        ),
        layer_thicknesses = [
            jnp.asarray(thickness)
            for thickness in thicknesses
        ]
    )

    # Use your existing result function
    sweep_result = (
        reflectance_and_transmittance_fmmax(
            s_matrix=sweep_s_matrix,
            layer_solve_results=sweep_layer_solve_result,
            polarization="s",
            thicknesses=thicknesses
        )
    )

    # R[0] is the result for the s-polarised incidence
    rs = (sweep_result["R"][0])

    actual_num_terms = int(sweep_expansion.num_terms)

    fmmax_rs_values.append(rs)
    fmmax_actual_orders.append(
        actual_num_terms
    )

    print(f"Requested: {requested_num_terms:3d}, "
          f"actual: {actual_num_terms:3d}," 
          f"Rs: {rs:.10f}"
          )

Requested:   5, actual:   5,Rs: 0.0214326250
Requested:  10, actual:   9,Rs: 0.0210720195
Requested:  15, actual:  13,Rs: 0.0209518992
Requested:  20, actual:  21,Rs: 0.0208197939
Requested:  25, actual:  21,Rs: 0.0208197939
Requested:  30, actual:  29,Rs: 0.0208051055
Requested:  35, actual:  37,Rs: 0.0207925277
Requested:  40, actual:  37,Rs: 0.0207925277
Requested:  45, actual:  45,Rs: 0.0207869043
Requested:  50, actual:  45,Rs: 0.0207869043
Requested:  55, actual:  57,Rs: 0.0206627132
Requested:  60, actual:  61,Rs: 0.0206625234
Requested:  65, actual:  69,Rs: 0.0206501901
Requested:  70, actual:  69,Rs: 0.0206501901
Requested:  75, actual:  69,Rs: 0.0206501901
Requested:  80, actual:  81,Rs: 0.0206360315
Requested:  85, actual:  89,Rs: 0.0206217464
Requested:  90, actual:  89,Rs: 0.0206217464
Requested:  95, actual:  97,Rs: 0.0206203143
Requested: 100, actual:  97,Rs: 0.0206203143


In [16]:
results_directory = Path("plots/csv")

results_directory.mkdir(
    parents=True,
    exist_ok=True
)

fmmax_output_path = (
    results_directory
    / "fmmax_fourier_convergence.csv"
)

with fmmax_output_path.open(
    "w",
    newline="",
) as file:

    writer = csv.writer(file)

    writer.writerow([
        "requested_orders",
        "actual_orders",
        "Rs",
    ])

    writer.writerows(
        zip(
            requested_orders,
            fmmax_actual_orders,
            fmmax_rs_values,
            strict=True,
        )
    )

    print(f"Saved results to {fmmax_output_path}")

Saved results to plots/csv/fmmax_fourier_convergence.csv
